In [3]:
# ==========================================================
# Import Libraries
# ==========================================================

from pyspark.sql import functions as F
from pyspark.sql.types import *

StatementMeta(, 9a151ef3-0ae7-4dbb-a1da-1cc80850f57f, 5, Finished, Available, Finished, False)

In [4]:
# ==========================================================
# Read Bronze Tables
# ==========================================================

customers = spark.table("bronze_customers")

products = spark.table("bronze_products")

stores = spark.table("bronze_stores")

employees = spark.table("bronze_employees")

sales = spark.table("bronze_sales")

returns = spark.table("bronze_returns")

StatementMeta(, 9a151ef3-0ae7-4dbb-a1da-1cc80850f57f, 6, Finished, Available, Finished, False)

In [5]:
# ==========================================================
# Validate Row Counts
# ==========================================================

tables = {

    "Customers": customers,

    "Products": products,

    "Stores": stores,

    "Employees": employees,

    "Sales": sales,

    "Returns": returns

}

for name, df in tables.items():

    print(f"{name:<12}: {df.count():,}")

StatementMeta(, 9a151ef3-0ae7-4dbb-a1da-1cc80850f57f, 7, Finished, Available, Finished, False)

Customers   : 1,000
Products    : 300
Stores      : 20
Employees   : 258
Sales       : 500,000
Returns     : 71,486


In [6]:
# ==========================================================
# Remove Duplicate Customers
# ==========================================================

customers = customers.dropDuplicates()

StatementMeta(, 9a151ef3-0ae7-4dbb-a1da-1cc80850f57f, 8, Finished, Available, Finished, False)

In [7]:
customers.count()

StatementMeta(, 9a151ef3-0ae7-4dbb-a1da-1cc80850f57f, 9, Finished, Available, Finished, False)

1000

In [8]:
# ==========================================================
# Standardize City Names
# ==========================================================

customers = customers.withColumn(

    "City",

    F.initcap(

        F.trim(

            F.col("City")

        )

    )

)

StatementMeta(, 9a151ef3-0ae7-4dbb-a1da-1cc80850f57f, 10, Finished, Available, Finished, False)

In [9]:
display(

    customers

    .groupBy("City")

    .count()

    .orderBy("City")

)

StatementMeta(, 9a151ef3-0ae7-4dbb-a1da-1cc80850f57f, 11, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 5d71d8a4-a513-410c-82ed-6f16abd92cdf)

In [10]:
# ==========================================================
# Fill Missing Emails
# ==========================================================

customers = customers.fillna(

    {

        "Email": "unknown@company.com"

    }

)

StatementMeta(, 9a151ef3-0ae7-4dbb-a1da-1cc80850f57f, 12, Finished, Available, Finished, False)

In [11]:
customers.filter(

    F.col("Email").isNull()

).count()

StatementMeta(, 9a151ef3-0ae7-4dbb-a1da-1cc80850f57f, 13, Finished, Available, Finished, False)

0

In [12]:
# ==========================================================
# Save Silver Customers
# ==========================================================

(

    customers.write

    .mode("overwrite")

    .format("delta")

    .saveAsTable(

        "silver_customers"

    )

)

StatementMeta(, 9a151ef3-0ae7-4dbb-a1da-1cc80850f57f, 14, Finished, Available, Finished, False)

In [13]:
spark.sql("""

SHOW TABLES

""").show(truncate=False)

StatementMeta(, 9a151ef3-0ae7-4dbb-a1da-1cc80850f57f, 15, Finished, Available, Finished, False)

+--------------------------------------+----------------+-----------+
|namespace                             |tableName       |isTemporary|
+--------------------------------------+----------------+-----------+
|`Retail Analytics`.RetailLakehouse.dbo|bronze_customers|false      |
|`Retail Analytics`.RetailLakehouse.dbo|bronze_employees|false      |
|`Retail Analytics`.RetailLakehouse.dbo|bronze_products |false      |
|`Retail Analytics`.RetailLakehouse.dbo|bronze_returns  |false      |
|`Retail Analytics`.RetailLakehouse.dbo|bronze_sales    |false      |
|`Retail Analytics`.RetailLakehouse.dbo|bronze_stores   |false      |
|`Retail Analytics`.RetailLakehouse.dbo|silver_customers|false      |
+--------------------------------------+----------------+-----------+



In [14]:
# ==========================================================
# Remove Duplicate Sales
# ==========================================================

sales = sales.dropDuplicates()

StatementMeta(, 9a151ef3-0ae7-4dbb-a1da-1cc80850f57f, 16, Finished, Available, Finished, False)

In [15]:
sales.count()

StatementMeta(, 9a151ef3-0ae7-4dbb-a1da-1cc80850f57f, 17, Finished, Available, Finished, False)

500000

In [16]:
# ==========================================================
# Fill Missing Payment Methods
# ==========================================================

sales = sales.fillna(

    {

        "PaymentMethod": "Unknown"

    }

)

StatementMeta(, 9a151ef3-0ae7-4dbb-a1da-1cc80850f57f, 18, Finished, Available, Finished, False)

In [17]:
sales.filter(

    F.col("PaymentMethod").isNull()

).count()

StatementMeta(, 9a151ef3-0ae7-4dbb-a1da-1cc80850f57f, 19, Finished, Available, Finished, False)

0

In [18]:
# ==========================================================
# Fill Missing Discount
# ==========================================================

sales = sales.fillna(

    {

        "Discount": 0

    }

)

StatementMeta(, 9a151ef3-0ae7-4dbb-a1da-1cc80850f57f, 20, Finished, Available, Finished, False)

In [19]:
sales.filter(

    F.col("Discount").isNull()

).count()

StatementMeta(, 9a151ef3-0ae7-4dbb-a1da-1cc80850f57f, 21, Finished, Available, Finished, False)

0

In [22]:
# ==========================================================
# Convert Date Columns
# ==========================================================

sales = sales.withColumn(
    "OrderDate",
    F.to_date("OrderDate")
)

StatementMeta(, 9a151ef3-0ae7-4dbb-a1da-1cc80850f57f, 24, Finished, Available, Finished, False)

In [23]:
sales.printSchema()

StatementMeta(, 9a151ef3-0ae7-4dbb-a1da-1cc80850f57f, 25, Finished, Available, Finished, False)

root
 |-- SaleID: integer (nullable = true)
 |-- OrderDate: date (nullable = true)
 |-- CustomerID: integer (nullable = true)
 |-- ProductID: integer (nullable = true)
 |-- EmployeeID: integer (nullable = true)
 |-- Quantity: integer (nullable = true)
 |-- Discount: integer (nullable = false)
 |-- PaymentMethod: string (nullable = false)
 |-- SalesChannel: string (nullable = true)
 |-- OrderStatus: string (nullable = true)
 |-- StoreID: integer (nullable = true)
 |-- UnitPrice: double (nullable = true)
 |-- UnitCost: integer (nullable = true)
 |-- CustomerCity: string (nullable = true)
 |-- StoreCity: string (nullable = true)
 |-- Revenue: double (nullable = true)
 |-- Cost: integer (nullable = true)
 |-- Profit: double (nullable = true)
 |-- Tax: double (nullable = true)



In [24]:
display(

    sales.select(

        "OrderDate",

        "Revenue",

        "Profit",

        "Discount"

    )

)

StatementMeta(, 9a151ef3-0ae7-4dbb-a1da-1cc80850f57f, 26, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, cb97835c-2e34-4c9a-8cf9-b1d291cc61df)

In [25]:
# ==========================================================
# Save Silver Sales
# ==========================================================

(
    sales.write
    .mode("overwrite")
    .format("delta")
    .saveAsTable("silver_sales")
)

StatementMeta(, 9a151ef3-0ae7-4dbb-a1da-1cc80850f57f, 27, Finished, Available, Finished, False)

In [26]:
spark.sql("SHOW TABLES").show(truncate=False)

StatementMeta(, 9a151ef3-0ae7-4dbb-a1da-1cc80850f57f, 28, Finished, Available, Finished, False)

+--------------------------------------+----------------+-----------+
|namespace                             |tableName       |isTemporary|
+--------------------------------------+----------------+-----------+
|`Retail Analytics`.RetailLakehouse.dbo|bronze_customers|false      |
|`Retail Analytics`.RetailLakehouse.dbo|bronze_employees|false      |
|`Retail Analytics`.RetailLakehouse.dbo|bronze_products |false      |
|`Retail Analytics`.RetailLakehouse.dbo|bronze_returns  |false      |
|`Retail Analytics`.RetailLakehouse.dbo|bronze_sales    |false      |
|`Retail Analytics`.RetailLakehouse.dbo|bronze_stores   |false      |
|`Retail Analytics`.RetailLakehouse.dbo|silver_customers|false      |
|`Retail Analytics`.RetailLakehouse.dbo|silver_sales    |false      |
+--------------------------------------+----------------+-----------+



In [27]:
# ==========================================================
# Clean Products
# ==========================================================

products = products.dropDuplicates()

StatementMeta(, 9a151ef3-0ae7-4dbb-a1da-1cc80850f57f, 29, Finished, Available, Finished, False)

In [28]:
products = products.withColumn(

    "ProductName",

    F.trim(

        F.col("ProductName")

    )

)

StatementMeta(, 9a151ef3-0ae7-4dbb-a1da-1cc80850f57f, 30, Finished, Available, Finished, False)

In [29]:
(
    products.write

    .mode("overwrite")

    .format("delta")

    .saveAsTable(

        "silver_products"

    )

)

StatementMeta(, 9a151ef3-0ae7-4dbb-a1da-1cc80850f57f, 31, Finished, Available, Finished, False)

In [30]:
# ==========================================================
# Clean Stores
# ==========================================================

stores = stores.dropDuplicates()

StatementMeta(, 9a151ef3-0ae7-4dbb-a1da-1cc80850f57f, 32, Finished, Available, Finished, False)

In [31]:
stores = stores.withColumn(

    "City",

    F.initcap(

        F.trim(

            F.col("City")

        )

    )

)

StatementMeta(, 9a151ef3-0ae7-4dbb-a1da-1cc80850f57f, 33, Finished, Available, Finished, False)

In [32]:
(
    stores.write

    .mode("overwrite")

    .format("delta")

    .saveAsTable(

        "silver_stores"

    )

)

StatementMeta(, 9a151ef3-0ae7-4dbb-a1da-1cc80850f57f, 34, Finished, Available, Finished, False)

In [33]:
# ==========================================================
# Clean Employees
# ==========================================================

employees = employees.dropDuplicates()

StatementMeta(, 9a151ef3-0ae7-4dbb-a1da-1cc80850f57f, 35, Finished, Available, Finished, False)

In [34]:
(
    employees.write

    .mode("overwrite")

    .format("delta")

    .saveAsTable(

        "silver_employees"

    )

)

StatementMeta(, 9a151ef3-0ae7-4dbb-a1da-1cc80850f57f, 36, Finished, Available, Finished, False)

In [35]:
# ==========================================================
# Clean Returns
# ==========================================================

returns = returns.dropDuplicates()

StatementMeta(, 9a151ef3-0ae7-4dbb-a1da-1cc80850f57f, 37, Finished, Available, Finished, False)

In [36]:
returns = returns.withColumn(

    "ReturnDate",

    F.to_date(

        "ReturnDate"

    )

)

StatementMeta(, 9a151ef3-0ae7-4dbb-a1da-1cc80850f57f, 38, Finished, Available, Finished, False)

In [37]:
(
    returns.write

    .mode("overwrite")

    .format("delta")

    .saveAsTable(

        "silver_returns"

    )

)

StatementMeta(, 9a151ef3-0ae7-4dbb-a1da-1cc80850f57f, 39, Finished, Available, Finished, False)

In [38]:
spark.sql("""

SHOW TABLES

""").show(truncate=False)

StatementMeta(, 9a151ef3-0ae7-4dbb-a1da-1cc80850f57f, 40, Finished, Available, Finished, False)

+--------------------------------------+----------------+-----------+
|namespace                             |tableName       |isTemporary|
+--------------------------------------+----------------+-----------+
|`Retail Analytics`.RetailLakehouse.dbo|bronze_customers|false      |
|`Retail Analytics`.RetailLakehouse.dbo|bronze_employees|false      |
|`Retail Analytics`.RetailLakehouse.dbo|bronze_products |false      |
|`Retail Analytics`.RetailLakehouse.dbo|bronze_returns  |false      |
|`Retail Analytics`.RetailLakehouse.dbo|bronze_sales    |false      |
|`Retail Analytics`.RetailLakehouse.dbo|bronze_stores   |false      |
|`Retail Analytics`.RetailLakehouse.dbo|silver_customers|false      |
|`Retail Analytics`.RetailLakehouse.dbo|silver_employees|false      |
|`Retail Analytics`.RetailLakehouse.dbo|silver_products |false      |
|`Retail Analytics`.RetailLakehouse.dbo|silver_returns  |false      |
|`Retail Analytics`.RetailLakehouse.dbo|silver_sales    |false      |
|`Retail Analytics`.